In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os 
import dotenv
import requests

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


In [17]:
dotenv.load_dotenv()
api_key = os.getenv("EIA_API_KEY")

python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5


In [29]:
def fetch_eia_hourly(api_key, endpoint, params):
    url = f"https://api.eia.gov/v2/{endpoint}/data/"
    params['api_key'] = api_key
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    df = pd.DataFrame(data.get('response', {}).get('data', []))
    if not df.empty:
        df['period'] = pd.to_datetime(df['period'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df

In [40]:
BA_list = ["BPAT", "PACW", "PSEI", "AVA", "CHPD", "GCPD", "DOPD", "SCL", "TPWR"]

In [49]:
hydro_params = {
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": BA_list,
    "facets[fueltype][]": ["WAT"],
    "start": "2026-08-01T00",
    "end": "2026-08-31T23",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}

load_params = {
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": BA_list,
    "facets[type][]": ["D"],
    "start": "2026-08-01T00",
    "end": "2026-08-31T23",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}

In [50]:
df_hydro = fetch_eia_hourly(
    api_key=api_key,
    endpoint="electricity/rto/fuel-type-data", 
    params=hydro_params
)

df_load = fetch_eia_hourly(
    api_key=api_key,
    endpoint="electricity/rto/region-data",
    params=load_params
)

In [51]:
df_load

,period,respondent,respondent-name,type,type-name,value,value-units
0,2026-08-01 00:00:00,AVA,Avista Corporation,D,Demand,2061,megawatthours
1,2026-08-01 00:00:00,BPAT,Bonneville Power Administration,D,Demand,8656,megawatthours
2,2026-08-01 00:00:00,CHPD,Public Utility District No. 1 of Chelan County,D,Demand,261,megawatthours
3,2026-08-01 00:00:00,DOPD,PUD No. 1 of Douglas County,D,Demand,357,megawatthours
4,2026-08-01 00:00:00,GCPD,"Public Utility District No. 2 of Grant County,...",D,Demand,969,megawatthours
...,...,...,...,...,...,...,...
4995,2026-08-24 03:00:00,AVA,Avista Corporation,D,Demand,1598,megawatthours
4996,2026-08-24 03:00:00,BPAT,Bonneville Power Administration,D,Demand,7754,megawatthours
4997,2026-08-24 03:00:00,CHPD,Public Utility District No. 1 of Chelan County,D,Demand,207,megawatthours
4998,2026-08-24 03:00:00,DOPD,PUD No. 1 of Douglas County,D,Demand,299,megawatthours


In [52]:
df_hydro_agg = df_hydro.groupby('period')['value'].sum().reset_index(name='total_hydro_mw')
df_load_agg = df_load.groupby('period')['value'].sum().reset_index(name='total_load_mw')